# Chapter 2 演習：評価指標の計算 (Calculating Evaluation Metrics)

この演習では、Numpyを使用してセマンティックセグメンテーションの出力結果をシミュレートし、**ピクセル精度 (Pixel Accuracy)** と **IoU (Intersection over Union)** を自分の手で計算します。

## シナリオ設定
自動運転のカメラ画像を想定します。画像サイズは非常にシンプルに $5 \times 5$ ピクセルとします。
クラスラベルは以下の2つです：
* `0`: 背景 (Background)
* `1`: 車 (Car)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. 真实标签 (Ground Truth) を定義
# 5x5の画像で、右下に車(1)があるとします。残りは背景(0)です。
ground_truth = np.array([
    [0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 0, 0, 1, 1],
    [0, 0, 0, 1, 1]
])

# 2. モデルの予測結果 (Prediction) を定義
# モデルは車の一部を見つけましたが、少しずれて予測してしまいました。
prediction = np.array([
    [0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 0, 0, 1, 0],
    [0, 0, 0, 1, 1],
    [0, 0, 0, 0, 0]
])

# 可視化関数 (Visualization)
def plot_masks(gt, pred):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4))
    ax1.imshow(gt, cmap='gray')
    ax1.set_title('Ground Truth (正解)')
    ax2.imshow(pred, cmap='gray')
    ax2.set_title('Prediction (予測)')
    plt.show()

plot_masks(ground_truth, prediction)

## 演習1：ピクセル精度 (Pixel Accuracy) の罠を体験する

Pixel Accuracyは「全体のうち、正解したピクセルの割合」です。
しかし、この指標は**クラス不均衡 (Class Imbalance)** の影響を受けやすいという問題があります。

**課題:**
すべてのピクセルを「背景 (0)」と予測するポンコツなモデル (`bad_prediction`) を作成しました。このモデルのPixel Accuracyを計算してください。

In [ ]:
# すべてを背景(0)と予測するモデル
bad_prediction = np.zeros((5, 5))

# ----------------------------------------------------
# TODO 1: bad_prediction と ground_truth が一致しているピクセルの数を計算してください。
# ヒント: np.sum(A == B) を使います。
correct_pixels = np.sum(bad_prediction == ground_truth)

# TODO 2: Pixel Accuracy を計算して表示してください。（正解数 / 全ピクセル数）
total_pixels = ground_truth.size
pixel_accuracy = correct_pixels / total_pixels
# ----------------------------------------------------

print(f"Bad Model Pixel Accuracy: {pixel_accuracy * 100}%")
# 考察：車を全く検出できていないのに、精度が高く見えませんか？これがクラス不均衡の問題です。

## 演習2：IoU (Intersection over Union) の計算

クラス不均衡の問題を解決するために、セグメンテーションでは **IoU** を使用します。
今回は「車 (クラス1)」に対するIoUを計算してみましょう。

計算式: 
$$IoU = \frac{\text{Intersection (重なり)}}{\text{Union (和集合)}} = \frac{TP}{TP + FP + FN}$$

* **TP (True Positive):** 正解が1で、予測も1の数
* **FP (False Positive):** 正解が0で、予測が1の数
* **FN (False Negative):** 正解が1で、予測が0の数

In [ ]:
# 対象クラス (車)
target_class = 1

# ----------------------------------------------------
# TODO 3: 最初の予測 (prediction) に対して、TP, FP, FN を計算してください。

# 真陽性 (True Positive): ground_truth と prediction が両方とも 1 のピクセル数
# ヒント: np.logical_and を使うか、ビット演算 `&` を使います。
TP = np.sum((ground_truth == target_class) & (prediction == target_class))

# 偽陽性 (False Positive): prediction が 1 なのに、ground_truth が 0 のピクセル数
FP = np.sum((ground_truth != target_class) & (prediction == target_class))

# 偽陰性 (False Negative): ground_truth が 1 なのに、prediction が 0 のピクセル数
FN = np.sum((ground_truth == target_class) & (prediction != target_class))

# TODO 4: IoU を計算してください。
iou = TP / (TP + FP + FN)
# ----------------------------------------------------

print(f"True Positives (TP): {TP}")
print(f"False Positives (FP): {FP}")
print(f"False Negatives (FN): {FN}")
print(f"IoU for Car Class: {iou:.4f}")